In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import scipy.stats as st

In [ ]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
amst_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_processed.csv")

In [ ]:
lnd_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_processed.csv")

In [ ]:
# Keep common columns only (defensive alignment)
shared_columns = [
    "person_id", "trip_id", "trip_purpose", "mode_detailed",
    "travel_time_min", "distance_km", "departure_hour",
    "n_legs", "n_transfers","has_transfer","weekday", "is_holiday",
    "age_band", "gender", "income_quintile", "has_driving_license",
    "n_cars_household", "weight_trip", "weight_person", "city",
    "is_peak", "chosen_mode"
]

In [ ]:
# Keep only needed columns for both datasets
amst_data = amst_data[[c for c in shared_columns if c in amst_data.columns]].copy()
lnd_data = lnd_data[[c for c in shared_columns if c in lnd_data.columns]].copy()

combined_data = pd.concat([amst_data, lnd_data], ignore_index = True) #Union both datasets

In [ ]:
# Harmonization
combined_data["chosen_mode"] = combined_data.chosen_mode.astype("category")
combined_data["city"] = combined_data.city.astype("category")

In [ ]:
# Duplicate rows check
combined_data.duplicated(subset=["city", "trip_id"]).sum()

In [ ]:
#Missing values
combined_data.isna().sum()

In [ ]:
# Check Invalid 
invalid_ranges = {
    "negative_time": (combined_data.travel_time_min <= 0).sum(),
    "negative_dist": (combined_data.distance_km <= 0).sum(),
    "invalid_hour": (~combined_data.departure_hour.between(0, 23)).sum(),
}

In [ ]:
invalid_ranges

In [ ]:
# Relationship/Logic checks
logic_errors = {
    "underage_commuters": (combined_data.age_band <= 3).sum(),  #Underage commuters are people in age band 1,2 and 3 (0-16 years old) 
    "transfer_mismatch": (combined_data.n_transfers != (combined_data.n_legs - 1)).sum(), # Transfer logic: n_transfers should usually be n_legs - 1
    "underage_driver": ((combined_data.age_band <= 3) & (combined_data.has_driving_license == True)).sum() # License check: if < 16-18 (depending on city), has_driving_license should be False
}

In [ ]:
logic_errors #we see that we have 35 trips which have more than 1 legs but no transfers

In [ ]:
combined_data[combined_data.n_transfers != (combined_data.n_legs - 1)][["chosen_mode","city"]].value_counts().sort_index()

In [ ]:
combined_data[combined_data.n_transfers != (combined_data.n_legs - 1)]

In [ ]:
combined_data.groupby("city").travel_time_min.describe().T.round(2)

In [ ]:
combined_data.groupby("city").distance_km.describe().T.round(2)

In [ ]:
# Categorical frequency snapshots
for variable in ["chosen_mode", "age_band", "income_quintile", "gender", "has_driving_license"]:
    freq_table = (combined_data[variable].value_counts(dropna = False).sort_index().to_frame("count") .assign(pct = lambda x: (x["count"] / x["count"].sum() * 100).round(1))
    )
    print(f"\nFrequency — {variable}")
    print(freq_table.head(10))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

city_colors = {"Amsterdam": "#FF9800", "London": "#2196F3"}

# Plot 1: Travel Time
sns.histplot(
    data = combined_data, x = "travel_time_min", hue = "city",
    bins = 30, stat = "density", common_norm = False, 
    palette = city_colors,
    alpha = 0.4, # Transparency (0 is clear, 1 is solid),
    # fill = True,
    ax = axes[0]
)
axes[0].set_title("Travel Time Distribution by City")
axes[0].set_xlabel("Travel time (min)")
axes[0].set_ylabel("Density")

# Plot 2: Distance
sns.histplot(
    data = combined_data, x = "distance_km", hue = "city",
    bins = 30, stat = "density", common_norm=False,
    palette = city_colors,
    alpha=0.4,
    # fill=True,
    ax=axes[1]
)
axes[1].set_title("Distance Distribution by City")
axes[1].set_xlabel("Distance (km)")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
# Unweighted mode share
mode_share_unweighted = (
    pd.crosstab(combined_data.city, combined_data.chosen_mode, normalize="index").mul(100).round(1))

In [ ]:
mode_share_unweighted

In [ ]:
# Weighted mode share using trip weights
weighted_counts = (
    combined_data.groupby(["city", "chosen_mode"], observed=False).weight_trip.sum().rename("weighted_sum").reset_index().round(0)
)

In [ ]:
weighted_counts

In [ ]:
weighted_totals = weighted_counts.groupby("city").weighted_sum.transform("sum")
weighted_counts["weighted_pct"] = (weighted_counts.weighted_sum / weighted_totals * 100).round(1)

mode_share_weighted = weighted_counts.pivot(index="city", columns="chosen_mode", values="weighted_pct").fillna(0)

In [ ]:
weighted_counts

In [ ]:
city_colors = {"Amsterdam": "#FF9800", "London": "#2196F3"}
chosen_modes = weighted_counts.chosen_mode.unique()


plt.figure(figsize=(10, 6))
sns.barplot(
    data = weighted_counts, 
    x = "chosen_mode", 
    y = "weighted_pct", 
    hue="city",
    palette = city_colors, # Using your city colors
    alpha=0.8
)

plt.title("Mode Share Comparison: Amsterdam vs. London", fontsize=14)
plt.xlabel("Transport Mode", fontsize=12)
plt.ylabel("Percentage of Total Trips (%)", fontsize=12)

plt.xticks(
    ticks = range(len(chosen_modes)), 
    labels = [str(m).upper() if m == 'pt' else str(m).title() for m in chosen_modes]
)

plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
alpha = 0.05

def run_chi2_test(data, var_a, var_b, alpha = 0.05):
    contingency = pd.crosstab(data[var_a], data[var_b])
    chi2_stat, p_value, dof, expected = st.chi2_contingency(contingency)
    expected_lt5 = (expected < 5).sum()
    total_cells = expected.size
    
    result = {
        "test": f"{var_a} vs {var_b}",
        "chi2": round(chi2_stat, 3),
        "dof": int(dof),
        "p_value": round(p_value, 6),
        "reject_h0": p_value < alpha,
        "expected_cells_lt5": int(expected_lt5),
        "total_cells": int(total_cells),
    }
    return contingency, result

chi2_pairs = [
    ("chosen_mode", "city"),
    ("chosen_mode", "income_quintile"),
    ("chosen_mode", "has_driving_license"),
    ("chosen_mode", "age_band"),
    ("chosen_mode", "is_peak")
]

chi2_results = []
for var_a, var_b in chi2_pairs:
    ctab, res = run_chi2_test(combined_data, var_a, var_b, alpha = alpha)
    chi2_results.append(res)
    print(f"\nContingency table: {var_a} x {var_b}")
    print(ctab)

chi2_results_table = pd.DataFrame(chi2_results)
print("\nChi-squared test summary")
print(chi2_results_table)

In [ ]:
def mode_groups(data, value_col):
    return [
        data.loc[data.chosen_mode == mode, value_col].dropna().values
        for mode in ["car", "pt", "bike"]
    ]

def run_anova_kruskal(data, value_col, alpha=0.05):
    groups = mode_groups(data, value_col)
    
    # Assumption checks
    # Levene for homogeneity
    levene_stat, levene_p = st.levene(*groups, center = "median")
    
    # ANOVA
    f_stat, f_p = st.f_oneway(*groups)
    
    # Kruskal
    h_stat, h_p = st.kruskal(*groups)
    
    # df for one-way ANOVA: k-1, N-k
    k = len(groups)
    n_total = sum(len(g) for g in groups)
    df_between = k - 1
    df_within = n_total - k
    
    result = {
        "variable": value_col,
        "levene_p": round(levene_p, 6),
        "anova_f": round(f_stat, 3),
        "anova_p": round(f_p, 6),
        "anova_df_between": df_between,
        "anova_df_within": df_within,
        "kruskal_h": round(h_stat, 3),
        "kruskal_p": round(h_p, 6),
        "anova_reject_h0": f_p < alpha,
        "kruskal_reject_h0": h_p < alpha
    }
    return result

test_vars = ["travel_time_min", "distance_km"]
continuous_test_results = pd.DataFrame([run_anova_kruskal(combined_data, v, alpha) for v in test_vars])
print(continuous_test_results)